# 03. Simulació del Mundial

Aquest notebook simula la Copa del Món amb els paràmetres entrenats previament.

In [1]:
import os
import sys
import csv
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import poisson

In [2]:
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.model import expected_goals
from src.elo import simulate_elos
from src.grid_search import csv_to_dict

In [3]:
import json
from pathlib import Path

project_root = Path.cwd().parent
json_path_1 = project_root / 'data' / 'raw' / 'team-conf.json'
json_path_2 = project_root / 'data' / 'raw' / 'conf-team.json'


with open(json_path_1, 'r', encoding='utf-8') as f:
    team_conf_dict = json.load(f)

with open(json_path_2, 'r', encoding='utf-8') as f:
    conf_team_dict = json.load(f)

In [4]:
df = pd.read_csv("../data/processed/df.csv")
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")

In [5]:
dict_k = csv_to_dict("../results/training/grid_search_train.csv")
with open('../results/training/model_params.csv', mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    # Extraemos la primera fila y convertimos los valores a float
    params = {k: float(v) for k, v in next(reader).items()}

In [6]:
df_matches, df_elos, df_confs = simulate_elos(df, team_conf_dict, dict_k, params)

In [7]:
def predict_match(home_team, away_team, current_elo_teams, current_elo_confs, params, neutral):
    # 1. Acceso instantáneo a los datos (sin .loc ni .iloc)
    h_data = current_elo_teams[home_team]
    a_data = current_elo_teams[away_team]

    # 2. Diferencia de Meta-ELO (usando el diccionario de confederaciones)
    conf_diff = current_elo_confs[h_data["Conf"]] - current_elo_confs[a_data["Conf"]]

    # 3. Ventaja de campo
    if neutral: # neutral ya es un booleano o 1/0
        h_adv, a_adv = 0, 0
    else:
        h_adv, a_adv = 1, -1

    # 4. Goles esperados (Lambda de Poisson)
    lam_home = expected_goals(params, h_data["Off_ELO"], a_data["Def_ELO"], h_adv, conf_diff)
    lam_away = expected_goals(params, a_data["Off_ELO"], h_data["Def_ELO"], a_adv, -conf_diff)

    # 5. Generar goles aleatorios
    goals_home = np.random.poisson(lam_home)
    goals_away = np.random.poisson(lam_away)

    return goals_home, goals_away, lam_home, lam_away

In [8]:
from src.groups import Group
import itertools

groups_data = {
    "A" : ["Mexico", "South Africa", "Korea Republic", "Czech Republic"],
    "B" : ["Canada", "Bosnia and Herzegovina", "Qatar", "Switzerland"],
    "C" : ["Brazil", "Morocco", "Haiti", "Scotland"],
    "D" : ["USA", "Paraguay", "Australia", "Turkey"],
    "E" : ["Germany", "Curaçao", "Côte d'Ivoire", "Ecuador"],
    "F" : ["Netherlands", "Japan", "Sweden", "Tunisia"],
    "G" : ["Belgium", "Egypt", "IR Iran", "New Zealand"],
    "H" : ["Spain", "Cape Verde", "Saudi Arabia", "Uruguay"],
    "I" : ["France", "Senegal", "Iraq", "Norway"],
    "J" : ["Argentina", "Algeria", "Austria", "Jordan"],
    "K" : ["Portugal", "Congo DR", "Uzbekistan", "Colombia"],
    "L" : ["England", "Croatia", "Ghana", "Panama"]
}

groups = {key: Group(key, teams) for key, teams in groups_data.items()}
teams_list = list(itertools.chain.from_iterable(groups_data.values()))

In [9]:
import random

def best_third(third_table, df_ranking):
    """
    Determina los 8 mejores terceros basándose en los criterios FIFA.
    
    Criterios:
    1. Puntos obtenidos
    2. Superior diferencia de goles
    3. Major nombre de gols marcats
    5. Random
    """
    
    # Preparamos una lista con la información necesaria para el desempate
    thirds_data = []
    
    for group_id, team_stats in third_table.items():
        # Extraemos el ranking FIFA del dataframe para este equipo
        # Nota: Usamos .item() para obtener el valor escalar del np.float64
        # ranking_value = df_ranking.loc[df_ranking['Team'] == team_stats['team'], 'ELO'].item()
        
        thirds_data.append({
            'group': group_id,
            'team': team_stats['team'],
            'points': team_stats['points'],
            'gd': team_stats['goal_difference'],
            'gf': team_stats['goals_scored'],
            # 'rank': ranking_value
        })

    # Ordenamos la lista aplicando la jerarquía de prioridades
    # Usamos reverse=True porque queremos de mayor a menor
    thirds_data.sort(key=lambda x: (
        x['points'], 
        x['gd'], 
        x['gf'], 
        # x['rank'], 
        random.random()
    ), reverse=True)

    # Seleccionamos los 8 mejores
    top_8_thirds = thirds_data[:8]
    
    letras_grupos = [item['group'] for item in top_8_thirds]
    
    letras_grupos.sort()
    
    return ",".join(letras_grupos)

In [10]:
def check_home_advantage(t1, t2):
    HOSTS_PRIORITY = ["USA", "Mexico", "Canada"]

    if t1 in HOSTS_PRIORITY and t2 not in HOSTS_PRIORITY:
        home_team = t1
        away_team = t2
        neutral = False
    elif t2 in HOSTS_PRIORITY and t1 not in HOSTS_PRIORITY:
        home_team = t2
        away_team = t1
        neutral = False
    elif t1 in HOSTS_PRIORITY and t2 in HOSTS_PRIORITY:
        if HOSTS_PRIORITY.index(t1) < HOSTS_PRIORITY.index(t2):
            home_team = t1
            away_team = t2
        else:
            home_team = t2
            away_team = t1
        neutral = False
    else:
        home_team = t1
        away_team = t2
        neutral = True
    return home_team, away_team, neutral

In [11]:
import copy

project_root = Path.cwd().parent

with open(project_root / 'data' / 'raw' / 'third_place.json', 'r', encoding='utf-8') as f:
    third_place_map = json.load(f)
with open(project_root / 'data' / 'raw' / 'tournament_bracket.json', 'r', encoding='utf-8') as f:
    bracket_data = json.load(f)


def simulate_world_cup_group_stage(n_simulations, df_ranking, df_confs, dict_k, params, groups, predict_match):
    initial_elo_teams = df_ranking.set_index('Team')[['Off_ELO', 'Def_ELO', 'Conf']].to_dict(orient='index')
    initial_elo_confs = dict(zip(df_confs['Confederation'], df_confs['Meta_ELO']))
    
    conf_members = {c: [t for t, info in initial_elo_teams.items() if info['Conf'] == c] 
                    for c in initial_elo_confs.keys()}
    
    group_stats = {g_label: {team: {1: 0, 2: 0, 3: 0, 4: 0} for team in g_obj.teams} 
                   for g_label, g_obj in groups.items()}
    
    for sim in range(n_simulations):
        current_elo_teams = copy.deepcopy(initial_elo_teams)
        current_elo_confs = copy.deepcopy(initial_elo_confs)

        days = {1: [], 2: [], 3: []}  
        for g_label, group_obj in groups.items():
            group_obj.reset()  
            teams = group_obj.teams
            days[1].append((g_label, teams[0], teams[1]))
            days[1].append((g_label, teams[2], teams[3]))
            days[2].append((g_label, teams[0], teams[2]))
            days[2].append((g_label, teams[1], teams[3]))
            days[3].append((g_label, teams[0], teams[3]))
            days[3].append((g_label, teams[1], teams[2]))
        
        for day in range(1, 4):
            # Simulate Group Stage Matches
            for g_label, t1, t2 in days[day]:
                h_team, a_team, neutral = check_home_advantage(t1, t2)
        
                g_h, g_a, exp_g_h, exp_g_a = predict_match(h_team, a_team, current_elo_teams, current_elo_confs, params, neutral)
                groups[g_label].update_match_result(h_team, a_team, g_h, g_a)
                diff_goals = abs(g_h - g_a)
                if diff_goals < 2: g = 1.0
                elif diff_goals == 2: g = 1.5
                else: g = (diff_goals + 11) / 8
                k = dict_k["k_0"]*dict_k["stage"]["Tournament"]*dict_k["competition"]["World_Cup"]

                current_elo_teams[h_team]["Off_ELO"] += k * g * (g_h - exp_g_h)
                current_elo_teams[a_team]["Def_ELO"] -= k * g * (g_h - exp_g_h)
                current_elo_teams[a_team]["Off_ELO"] += k * g * (g_a - exp_g_a)
                current_elo_teams[h_team]["Def_ELO"] -= k * g * (g_a - exp_g_a)

                conf_h = current_elo_teams[h_team]['Conf']
                conf_a = current_elo_teams[a_team]['Conf']

                if conf_h != conf_a:
                    err_h = g_h - exp_g_h
                    err_a = g_a - exp_g_a
                    omega = err_h - err_a
        
                    delta = g * k * dict_k['eta'] * omega
                    current_elo_confs[conf_h] += delta
                    current_elo_confs[conf_a] -= delta
                    
                    # Ripple Effect
                    for team in conf_members[conf_h]:
                        current_elo_teams[team]["Off_ELO"] += delta
                        current_elo_teams[team]["Def_ELO"] += delta
                    for team in conf_members[conf_a]:
                        current_elo_teams[team]["Off_ELO"] -= delta
                        current_elo_teams[team]["Def_ELO"] -= delta

        for g_label, g_obj in groups.items():
            table = g_obj.get_table(current_elo_teams)
            for pos, row in enumerate(table, 1): # pos va de 1 a 4
                team_name = row['team']
                group_stats[g_label][team_name][pos] += 1
            # qualified_from_groups[f"1{g}"] = table[0]['team']
            # qualified_from_groups[f"2{g}"] = table[1]['team']
            # third_table[g] = table[2]
        # thirds_classify = best_third(third_table, df_ranking)
        # third_assignments = third_place_map[thirds_classify]
        # print(qualified_from_groups)
    return group_stats

In [13]:
n_simulations = 100000
group_stats = simulate_world_cup_group_stage(n_simulations, df_elos, df_confs, dict_k, params, groups, predict_match)

KeyboardInterrupt: 

In [59]:
probabilidades_por_grupo = {}

for g_label, equipos in group_stats.items():
    # 1. Convertimos a DataFrame
    df = pd.DataFrame(equipos).T
    
    # 2. Dividimos todos los valores por 1000 y multiplicamos por 100 para tener el %
    # Nota: Si tus datos suman 3000, esto te dará valores como 100.6%. 
    # Si quieres el % real sobre el total actual, divide por 3000.
    df_prob = (df / n_simulations) * 100 
    
    # 3. Renombramos las columnas para que sean claras
    df_prob.columns = ['1º (%)', '2º (%)', '3º (%)', '4º (%)']
    
    probabilidades_por_grupo[g_label] = df_prob.round(2)

for g_label, df_prob in probabilidades_por_grupo.items():
    print(f"\nGrupo {g_label}:\n{df_prob}\n")


Grupo A:
                1º (%)  2º (%)  3º (%)  4º (%)
Mexico           27.61   27.85   25.05   19.49
South Africa     12.92   19.28   27.12   40.68
Korea Republic   26.29   26.04   25.36   22.31
Czech Republic   33.17   26.83   22.48   17.52


Grupo B:
                        1º (%)  2º (%)  3º (%)  4º (%)
Canada                   17.56   38.90   30.94   12.60
Bosnia and Herzegovina   10.88   30.65   39.34   19.12
Qatar                     2.02    8.62   22.73   66.63
Switzerland              69.54   21.83    6.98    1.65


Grupo C:
          1º (%)  2º (%)  3º (%)  4º (%)
Brazil     69.36   24.30    6.02    0.32
Morocco    24.28   50.94   21.97    2.80
Haiti       0.20    2.09   14.14   83.57
Scotland    6.15   22.67   57.87   13.31


Grupo D:
           1º (%)  2º (%)  3º (%)  4º (%)
USA         29.03   26.55   23.73   20.68
Paraguay    18.72   23.50   27.42   30.37
Australia   19.50   23.71   26.85   29.93
Turkey      32.75   26.24   21.99   19.01


Grupo E:
               1º (%)

In [14]:
def update_elos_tournament(h_team, a_team, h_goals, a_goals, exp_h_goals, exp_a_goals, current_elo_teams, current_elo_confs, dict_k, conf_members):
    diff_goals = abs(h_goals - a_goals)
    g_fact = 1.0 if diff_goals < 2 else 1.5 if diff_goals == 2 else (diff_goals + 11) / 8
    k = dict_k["k_0"]*dict_k["stage"]["Tournament"]*dict_k["competition"]["World_Cup"]

    current_elo_teams[h_team]["Off_ELO"] += k * g_fact * (h_goals - exp_h_goals)
    current_elo_teams[h_team]["Def_ELO"] += k * g_fact * (a_goals - exp_a_goals) # Mejora si encaja menos de lo esperado
    current_elo_teams[a_team]["Off_ELO"] += k * g_fact * (a_goals - exp_a_goals)
    current_elo_teams[a_team]["Def_ELO"] += k * g_fact * (h_goals - exp_h_goals)

    conf_h, conf_a = current_elo_teams[h_team]['Conf'], current_elo_teams[a_team]['Conf']
    if conf_h != conf_a:
        omega = (h_goals - exp_h_goals) - (a_goals - exp_a_goals)
        delta = g_fact * k * dict_k['eta'] * omega
        current_elo_confs[conf_h] += delta
        current_elo_confs[conf_a] -= delta
        for team in conf_members[conf_h]:
            current_elo_teams[team]["Off_ELO"] += delta
            current_elo_teams[team]["Def_ELO"] += delta
        for team in conf_members[conf_a]:
            current_elo_teams[team]["Off_ELO"] -= delta
            current_elo_teams[team]["Def_ELO"] -= delta
    
    return current_elo_teams, current_elo_confs

In [15]:
def simulate_world_cup(n_simulations, df_ranking, df_confs, params, groups, predict_match, dict_k, third_place_map, bracket_data, results,teams_list):
    team_to_idx = {team: i for i, team in enumerate(teams_list)}
    
    # Matriz para guardar la ronda alcanzada (1 a 7)
    results_matrix = np.zeros((n_simulations, len(teams_list)))

    # initial_elo_teams = df_ranking.set_index('Team')[['ELO', 'Off_ELO', 'Def_ELO', 'Conf']].to_dict(orient='index')
    initial_elo_teams = df_ranking.set_index('Team')[['Off_ELO', 'Def_ELO', 'Conf']].to_dict(orient='index')
    initial_elo_confs = dict(zip(df_confs['Confederation'], df_confs['Meta_ELO']))
    
    conf_members = {c: [t for t, info in initial_elo_teams.items() if info['Conf'] == c] 
                    for c in initial_elo_confs.keys()}
        

    for sim in range(n_simulations):
        current_elo_teams = copy.deepcopy(initial_elo_teams)
        current_elo_confs = copy.deepcopy(initial_elo_confs)
        third_table = {}
        qualified_from_groups = {}

        for team in teams_list:
            results_matrix[sim, team_to_idx[team]] = 8

        # 1. FASE DE GRUPOS
        days = {1: [], 2: [], 3: []}  
        for g_label, group_obj in groups.items():
            group_obj.reset()  
            teams = group_obj.teams
            days[1].append((g_label, teams[0], teams[1]))
            days[1].append((g_label, teams[2], teams[3]))
            days[2].append((g_label, teams[0], teams[2]))
            days[2].append((g_label, teams[1], teams[3]))
            days[3].append((g_label, teams[0], teams[3]))
            days[3].append((g_label, teams[1], teams[2]))
        
        for day in range(1, 4):
            for g_label, t1, t2 in days[day]:
                h_team, a_team, neutral = check_home_advantage(t1, t2)
                g_h, g_a, exp_g_h, exp_g_a = predict_match(h_team, a_team, current_elo_teams, current_elo_confs, params, neutral)

                if results['groups'][g_label][f"{h_team} vs {a_team}"] != -1:
                    g_h, g_a = results['groups'][g_label][f"{h_team} vs {a_team}"]

                groups[g_label].update_match_result(h_team, a_team, g_h, g_a)

                current_elo_teams, current_elo_confs = update_elos_tournament(
                    h_team, a_team, g_h, g_a, exp_g_h, exp_g_a, current_elo_teams, current_elo_confs, dict_k, conf_members)

        for g_label, g_obj in groups.items():
            table = g_obj.get_table(current_elo_teams)
            qualified_from_groups[f"1{g_label}"] = table[0]['team']
            qualified_from_groups[f"2{g_label}"] = table[1]['team']
            third_table[g_label] = table[2]
            
        thirds_classify = best_third(third_table, df_ranking)
        third_assignments = third_place_map[thirds_classify]

        for match_id, third_code in third_assignments.items():
            group_letter = third_code[1]
            qualified_from_groups[third_code] = third_table[group_letter]['team']

        # 3. ELIMINATORIAS
        results_knockout = {}

        rounds_config = [
            ("round_of_32", 7),
            ("round_of_16", 6),
            ("quarter_finals", 5),
            ("semi_finals", 4),
            ("final", 1)
        ]

        third_and_fourth_place = []        
        
        for r_name, r_level in rounds_config:
            for mid, teams in bracket_data[r_name].items():
                if r_name == "round_of_32":
                    t1_code = teams[0]
                    t2_code = teams[1] if teams[1] != "" else third_assignments[mid]
                    team_a, team_b = qualified_from_groups[t1_code], qualified_from_groups[t2_code]
                else:
                    team_a, team_b = results_knockout[teams[0]], results_knockout[teams[1]]
                
                h_team, a_team, neutral = check_home_advantage(team_a, team_b)
                
                g_h, g_a, exp_g_h, exp_g_a = predict_match(h_team, a_team, current_elo_teams, current_elo_confs, params, neutral)

                if results['knockout'][r_name][mid] != -1:
                    g_h = results['knockout'][r_name][mid]["home_goals"]
                    g_a = results['knockout'][r_name][mid]["away_goals"]
                    winner = results['knockout'][r_name][mid]["winner"]
                    losser = a_team if winner == h_team else h_team
                else:
                    if g_h > g_a: winner = h_team
                    elif g_a > g_h: winner = a_team
                    else: winner = random.choice([h_team, a_team])
                    
                    losser = a_team if winner == h_team else h_team

                results_knockout[mid] = winner

                if r_name in ["round_of_32", "round_of_16", "quarter_finals"]:
                    results_matrix[sim, team_to_idx[losser]] = r_level

                if r_name == "semi_finals":
                    third_and_fourth_place.append(losser)
                    if len(third_and_fourth_place) == 2:
                        t1, t2 = third_and_fourth_place
                        h_team, a_team, neutral = check_home_advantage(t1, t2)
                        g_h, g_a, exp_g_h, exp_g_a = predict_match(h_team, a_team, current_elo_teams, current_elo_confs, params, neutral)

                        if g_h > g_a: winner = h_team
                        elif g_a > g_h: winner = a_team
                        else: winner = random.choice([h_team, a_team])

                        losser = a_team if winner == h_team else h_team

                        results_matrix[sim, team_to_idx[losser]] = 4
                        results_matrix[sim, team_to_idx[winner]] = 3

                        current_elo_teams, current_elo_confs = update_elos_tournament(h_team, a_team, g_h, g_a, exp_g_h, exp_g_a, current_elo_teams, current_elo_confs, dict_k, conf_members)

                if r_name == "final":
                    results_matrix[sim, team_to_idx[losser]] = 2
                    results_matrix[sim, team_to_idx[winner]] = 1

                current_elo_teams, current_elo_confs = update_elos_tournament(h_team, a_team, g_h, g_a, exp_g_h, exp_g_a, current_elo_teams, current_elo_confs, dict_k, conf_members)
    
    return results_matrix, teams_list

In [16]:
project_root = Path.cwd().parent

with open(project_root / 'data' / 'raw' / 'third_place.json', 'r') as f:
    third_place_map = json.load(f)

with open(project_root / 'data' / 'raw' / 'tournament_bracket.json', 'r') as f:
    bracket_data = json.load(f)

with open(project_root / 'data' / 'raw' / 'results.json', 'r', encoding="utf-8") as f:
    results = json.load(f)


n_simulations = 100000
simulation_results, teams = simulate_world_cup(n_simulations, df_elos, df_confs, params, groups, predict_match, dict_k, third_place_map, bracket_data, results, teams_list)

In [17]:
df_simulaciones = pd.DataFrame(simulation_results, columns=teams_list)
df_simulaciones.to_csv("../results/simulation/simulation_1st_day.csv", index=False, encoding="utf-8")

In [18]:
def calculate_tournament_probabilities(results_matrix, teams_list):
    # Definimos los niveles de ronda (según la lógica de la matriz)
    # 1:Grupos, 2:R32, 3:R16, 4:QF, 5:SF, 6:Finalista, 7:Campeón
    n_sims = results_matrix.shape[0]
    
    prob_data = []
    for i, team in enumerate(teams_list):
        team_results = results_matrix[:, i]
        
        prob_data.append({
            "Selección": team,
            "Llegar a R32 (%)": (np.sum(team_results < 8) / n_sims) * 100, # Superó grupos, llegó a R32
            "Llegar a Octavos (%)":      (np.sum(team_results < 7) / n_sims) * 100, # Superó R32, llegó a R16
            "Llegar a Cuartos (%)":      (np.sum(team_results < 6) / n_sims) * 100, # Superó R16, llegó a QF
            "Llegar a Semis (%)":        (np.sum(team_results < 5) / n_sims) * 100, # Superó QF, llegó a SF
            "Llegar a Final (%)":        (np.sum(team_results < 3) / n_sims) * 100, # Superó SF, llegó a la Final
            "Campeón (%)":      (np.sum(team_results == 1) / n_sims) * 100  # Ganó la Final
        })

    df_probs = pd.DataFrame(prob_data)
    return df_probs.sort_values(by="Campeón (%)", ascending=False).reset_index(drop=True)

# Ejecución
df_resultados = pd.read_csv("../results/simulation/simulation_1st_day.csv")
teams = df_resultados.columns.tolist()
matriz_resultados = df_resultados.to_numpy()
df_final_probs = calculate_tournament_probabilities(matriz_resultados, teams)
# df_final_probs = calculate_tournament_probabilities(simulation_results, teams)

In [19]:
df_final_probs

,Selección,Llegar a R32 (%),Llegar a Octavos (%),Llegar a Cuartos (%),Llegar a Semis (%),Llegar a Final (%),Campeón (%)
0,England,99.915,80.558,67.008,43.087,27.503,16.312
1,Spain,97.310,71.510,51.600,38.852,24.679,15.104
2,France,99.697,79.220,48.624,32.051,18.956,10.422
3,Brazil,99.439,66.352,47.566,27.539,16.371,9.503
4,Argentina,99.792,65.069,49.060,31.261,16.882,9.397
5,Portugal,94.466,69.125,45.403,28.629,15.121,7.880
6,Germany,99.857,77.565,43.641,28.439,15.707,7.762
7,Netherlands,91.022,53.075,38.656,19.268,10.117,4.727
8,Belgium,95.562,64.981,43.610,20.648,9.728,4.133
9,Colombia,98.851,61.222,34.678,17.538,7.537,3.132
